In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bacco

In [ ]:
def read_cpu(filename=None, skiprows=[]):
    with open(filename) as f:
        lines = f.readlines()

    d = {}

    i=0
    for line in lines:
        line.strip()

        if i == 0:
            columns = [item.strip() for item in line.split(',')]
            for index, elem in enumerate(columns):
                d[columns[index]] = []
        elif i in skiprows:
            i = i + 1
            continue
        else:
            data = [item.strip() for item in line.split(',')]
            for index, elem in enumerate(columns):
                if index < 8:
                    d[columns[index]].append(float(data[index]))
                else:
                    d[columns[index]].append(float(data[index+1]))
        print(i)
        i = i + 1

    return d

In [ ]:
# Read the file and extract the data for the given CPU name prefix and columns
def extract_data(filename):
    with open(filename, 'r') as file:
        reader = csv.reader(file, delimiter=',')
        header = next(reader)  # Skip the header row
        relevant_index = [i for i, col_name in enumerate(header)]
        data = {header[i]: [] for i, col_name in enumerate(header)}

        for row in reader:
            try:
                for i,col_name in enumerate(header):
                    data[col_name].append(float(row[i]))
            except:
                continue
    
    return data

In [ ]:
def read_csv(filename):
    i=0
    with open(filename) as f:
        data = []
        lines = f.readlines()
        for line in lines:
            if i==51232 or i==137030 or i==20437:
                i+=1
                continue
            else:
                try:
                    data.append(float(line.strip()))
                    i+=1
                except:
                    print(i)
    return np.array(data)

In [ ]:
def load_scaling(base, norm=1):
    data = {}

    data['step'] = read_csv(base+'step.csv')
    data['expfactor'] = read_csv(base+'expfactor.csv')
    data['diff_cpu'] = read_csv(base+'diff_cpu.csv') / 3600 * norm
    data['total_cpu'] = read_csv(base+'total_cpu.csv') / 3600 * norm
    data['tree_cpu'] = read_csv(base+'tree_cpu.csv') / 3600 * norm
    data['tree_walk_cpu'] = read_csv(base+'tree_walk_cpu.csv') / 3600 * norm
    data['tree_bal_cpu'] = read_csv(base+'tree_bal_cpu.csv') / 3600 * norm
    data['pm_cpu'] = read_csv(base+'pm_cpu.csv') / 3600 * norm
    data['voronoi_cpu'] = read_csv(base+'voronoi_cpu.csv') / 3600 * norm
    data['hydro_cpu'] = read_csv(base+'hydro_cpu.csv') / 3600 * norm
    data['gfm_cpu'] = read_csv(base+'gfm_cpu.csv') / 3600 * norm
    data['restart_cpu'] = read_csv(base+'restart_cpu.csv') / 3600 * norm

    return data

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_0/hydro_output/"
data_fid = load_scaling(base, norm=1008)

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

w1 = np.where(data_fid['expfactor']<0.334)
w2 = np.where(data_fid['expfactor']>0.334)

ax.plot(data_fid['step'][w1], data_fid['diff_cpu'][w1], color='r', lw=1, ls='', marker='.', ms=0.3)
ax.plot(data_fid['step'][w2], data_fid['diff_cpu'][w2], color='b', lw=1, ls='', marker='.', ms=0.3)

#ax.set_xlim(0,1)

ax.legend(loc='upper left')

ax.set_xlabel('$a$')
ax.set_ylabel('CPU-hours per timestep')

In [ ]:
fig, ax = plt.subplots(dpi=200)

ax.set_xscale('log')

ax.hist(data_fid['diff_cpu'][w1], bins=np.logspace(-2,3,100), log=True, histtype='step', color='r')
ax.hist(data_fid['diff_cpu'][w2], bins=np.logspace(-2,3,100), log=True, histtype='step', color='b')

ax.set_xlabel('CPU-hours')
ax.set_ylabel('Counts')

In [ ]:
def plot_cumulative(data, ax, len):
    keys = np.array(list(data.keys()))

    mask = data['expfactor'][:len] > 0
    
    for key in keys:
        data[key] = data[key][:len][mask]

    ax.grid()
    ax.fill_between(data['expfactor'], np.zeros_like(data['total_cpu']), data['total_cpu'], color=colors[6], label='Total')
    ax.fill_between(data['expfactor'], np.zeros_like(data['total_cpu']), data['hydro_cpu'], label='Hydro', color=colors[0])
    ax.fill_between(data['expfactor'], data['hydro_cpu'],\
                                             data['hydro_cpu'] + data['voronoi_cpu'][:len][mask], label='Voronoi', color=colors[1])
    ax.fill_between(data['expfactor'][:len][mask], data['hydro_cpu']+ data['voronoi_cpu'],\
                                             data['hydro_cpu']+ data['voronoi_cpu'] + data['pm_cpu'], label='PM', color=colors[2])
    ax.fill_between(data['expfactor'][:len][mask], data['hydro_cpu']+ data['voronoi_cpu'] + data['pm_cpu'],
                                             data['hydro_cpu']+ data['voronoi_cpu'] + data['pm_cpu'] + data['tree_cpu'], label='Tree-Grav', color=colors[3])
    ax.fill_between(data['expfactor'][:len][mask], data['hydro_cpu']+ data['voronoi_cpu'] + data['pm_cpu'] + data['tree_cpu'], \
                                             data['hydro_cpu']+ data['voronoi_cpu'] + data['pm_cpu'] + data['tree_cpu'] + data['gfm_cpu'], label='GFM', color=colors[4])
    ax.fill_between(data['expfactor'][:len][mask], data['hydro_cpu']+ data['voronoi_cpu'] + data['pm_cpu'] + data['tree_cpu'] + data['gfm_cpu'], \
                                             data['hydro_cpu']+ data['voronoi_cpu'] + data['pm_cpu'] + data['tree_cpu'] + data['gfm_cpu'] + data['restart_cpu'], label='Restart', color=colors[5])


In [ ]:
fig, ax = plt.subplots(1, dpi=200, figsize=(5.5, 5), sharey=True, sharex=True)

# ax.set_xscale('log')
# ax.set_yscale('log')

ax.set_xlim(1e-2,1)

colors=[ '#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377', '#BBBBBB']

plot_cumulative(data_fid, ax, 1007572)

ax.legend(loc='upper left')

### Now let's compare the scaling of different number of halos

Get total mass of each selection

In [ ]:
sel_10 = np.loadtxt('/cosmos_storage/data_sharing/MN5_resims/level4/PM_1536/halos_10/vmax_custom_sel_10.txt', int)
sel_100 = np.loadtxt('/cosmos_storage/data_sharing/MN5_resims/level4/PM_1536/halos_100/vmax_custom_sel_100.txt', int)
sel_1000 = np.loadtxt('/cosmos_storage/data_sharing/MN5_resims/level4/PM_1536/halos_1000/vmax_custom_sel_1000.txt', int)

In [ ]:
basedir = "/cosmos_storage/simulations/MTNG/DM-Gadget4/MTNG-L500-1080-A/"

resolution_level = 1

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME
numpart = int(1080**3*2**(3-3*resolution_level))

mtng_dm = bacco.Simulation(basedir=basedir, halo_file="groups_264/fof_subhalo_tab_264", sim_format='TNG500', fixedPk=True, sigma8=sigma8,\
    tau=tau, ns=ns, numpart=numpart, use_orphans=False)

In [ ]:
m_10 = np.sum(mtng_dm.fof['halo_m200b'][sel_10])
m_100 = np.sum(mtng_dm.fof['halo_m200b'][sel_100])
m_1000 = np.sum(mtng_dm.fof['halo_m200b'][sel_1000])

In [ ]:
m_1000 / 1e5

In [ ]:
0.15 * m_1000

In [ ]:
100 * (m_1000 / M_T )

In [ ]:
M_T = np.sum(mtng_dm.fof['halo_m200b'])

In [ ]:
# base = "/cosmos_storage/data_sharing/MN5_resims/PM_1024/halos_10/"
# data_10_1024 = load_scaling(base, norm=112/m_10)

base = "/cosmos_storage/data_sharing/MN5_resims/PM_1536/halos_10/"
data_10_1536 = load_scaling(base, norm=112/m_10)

base = "/cosmos_storage/data_sharing/MN5_resims/PM_1536/halos_100/"
data_100_1536 = load_scaling(base, norm=560/m_100)

base = "/cosmos_storage/data_sharing/MN5_resims/PM_1792/halos_100/"
data_100_1792 = load_scaling(base, norm=1120/m_100)

base = "/cosmos_storage/data_sharing/MN5_resims/PM_1536/halos_1000/"
data_1000_1536 = load_scaling(base,  norm=560/m_1000)

base = "/cosmos_storage/data_sharing/MN5_resims/PM_2048/halos_1000/"
data_1000_2048 = load_scaling(base,  norm=1120/m_1000)

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
#ax.set_yscale('log')

ax.plot(data_10_1024['expfactor'], data_10_1024['total_cpu'], color=colors[0], label='$M_f={:.2f}$'.format(m_10/M_T/1e-5) +'$\\times 10^{-5},$  $PM=1024$')
ax.plot(data_100_1792['expfactor'], data_100_1792['total_cpu'], color=colors[2], label='$M_f={:.2f}$'.format(m_100/M_T/1e-4) +'$\\times 10^{-4},$  $PM=1729$')
ax.plot(data_1000_2048['expfactor'], data_1000_2048['total_cpu'][:26793], color=colors[4], label='$M_f={:.2f}$'.format(m_1000/M_T/1e-3) +'$\\times 10^{-3},$  $PM=2048$')

#ax.set_ylim(0,50)

ax.legend(loc='upper left')

ax.set_xlabel('$a$')
ax.set_ylabel('CPU-hours$/(10^{10}M_{\odot}h^{-1})$')

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
#ax.set_yscale('log')

ax.plot(data_10_1536['expfactor'], data_10_1536['total_cpu'], color=colors[0], label='$M_f={:.2f}$'.format(m_10/M_T/1e-5) +'$\\times 10^{-5},$  $PM=1536$')
ax.plot(data_100_1536['expfactor'], data_100_1536['total_cpu'], color=colors[2], label='$M_f={:.2f}$'.format(m_100/M_T/1e-4) +'$\\times 10^{-4},$  $PM=1536$')
ax.plot(data_1000_1536['expfactor'], data_1000_1536['total_cpu'][:26793], color=colors[4], label='$M_f={:.2f}$'.format(m_1000/M_T/1e-3) +'$\\times 10^{-3},$  $PM=1536$')

#ax.set_ylim(0,50)

ax.legend(loc='upper left')

ax.set_xlabel('$a$')
ax.set_ylabel('CPU-hours$/(10^{10}M_{\odot}h^{-1})$')

In [ ]:
fig, ax = plt.subplots(1, 3, dpi=200, figsize=(15, 5), sharey=True, sharex=True)

ax[0].set_xscale('log')
ax[1].set_xscale('log')

ax[0].set_xlim(1e-2, 1)
ax[0].set_ylim(-0.005,0.12)

#ax.set_yscale('log')

colors=[ '#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377', '#BBBBBB']

plot_cumulative(data_10_1024, ax[0], 232186)
ax[0].set_title('$M_f={:.2f}$'.format(m_10/M_T/1e-5) +'$\\times 10^{-5},$  $PM=1024$')

plot_cumulative(data_100_1792, ax[1], 31573)
ax[1].set_title(label='$M_f={:.2f}$'.format(m_100/M_T/1e-4) +'$\\times 10^{-4},$  $PM=1792$')

plot_cumulative(data_1000_2048, ax[2], 26791)
ax[2].set_title(label='$M_f={:.2f}$'.format(m_1000/M_T/1e-3) +'$\\times 10^{-3},$  $PM=2048$')

ax[0].set_xlabel('$a$')
ax[0].set_ylabel('CPU-hours$/(10^{10}M_{\odot}h^{-1})$')

ax[1].set_xlabel('$a$')

ax[0].legend(loc='upper left')

In [ ]:
fig, ax = plt.subplots(1, 3, dpi=200, figsize=(15, 5), sharey=True, sharex=True)

ax[0].set_xscale('log')
ax[1].set_xscale('log')

ax[0].set_xlim(1e-2, 1)
ax[0].set_ylim(-0.005,0.12)

colors=[ '#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377', '#BBBBBB']

plot_cumulative(data_10_1536, ax[0], 131939)
ax[0].set_title('$M_f={:.2f}$'.format(m_10/M_T/1e-5) +'$\\times 10^{-5},$  $PM=1536$')

plot_cumulative(data_100_1536, ax[1], 84345)
ax[1].set_title(label='$M_f={:.2f}$'.format(m_100/M_T/1e-4) +'$\\times 10^{-4},$  $PM=1536$')

plot_cumulative(data_1000_1536, ax[2], 762)
ax[2].set_title(label='$M_f={:.2f}$'.format(m_1000/M_T/1e-3) +'$\\times 10^{-3},$  $PM=1536$')

ax[0].set_xlabel('$a$')
ax[0].set_ylabel('CPU-hours$/(10^{10}M_{\odot}h^{-1})$')

ax[1].set_xlabel('$a$')

ax[0].legend(loc='upper left')

In [ ]:
fig, ax = plt.subplots(1, 2, dpi=200, figsize=(10, 5), sharey=True, sharex=True)

ax[0].set_xscale('log')
ax[1].set_xscale('log')

colors=[ '#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377', '#BBBBBB']

plot_cumulative(data_1000_1536, ax[0], 762)
ax[0].set_title('$PM=1536$')

plot_cumulative(data_1000_2048, ax[1], 12284)
ax[1].set_title('$PM=2048$')

ax[0].set_xlabel('$a$')
ax[0].set_ylabel('CPU-hours/Halo')

ax[1].set_xlabel('$a$')

ax[0].legend(loc='upper left')